<a href="https://colab.research.google.com/github/vifirsanova/nlp-course/blob/main/Transformers/BERT-WNLI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Настройка BERT под задачу WNLI

**Цель работы:** научиться дообучать (fine-tune) предобученную модель BERT для решения конкретной задачи классификации текстов из набора данных GLUE на примере задачи Winograd Natural Language Inference (WNLI).

**Важно:** перед началом работы убедитесь, что в среде выполнения включена поддержка GPU (Среда выполнения → Сменить среду выполнения → Аппаратный ускоритель: GPU).

#### Импорт библиотек

In [1]:
import datasets
import random
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import transformers

#### Выбор задачи и модели

In [2]:
task = "wnli"
model_checkpoint = "bert-base-multilingual-cased"
batch_size = 16

**Задание 1:** замените `"wnli"` на `"cola"` или `"mrpc"`. Что изменится в процессе загрузки данных и выборе метрик? Ищите ответ в словаре `task_to_keys` и в [документации GLUE](https://huggingface.co/datasets/glue).

#### Загрузка данных

In [3]:
from datasets import load_dataset

dataset = load_dataset("glue", task)

dataset

README.md: 0.00B [00:00, ?B/s]

wnli/train-00000-of-00001.parquet:   0%|          | 0.00/38.8k [00:00<?, ?B/s]

wnli/validation-00000-of-00001.parquet:   0%|          | 0.00/11.1k [00:00<?, ?B/s]

wnli/test-00000-of-00001.parquet:   0%|          | 0.00/13.6k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/635 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/71 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/146 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 635
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 71
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 146
    })
})

In [4]:
dataset["train"][0]

{'sentence1': 'I stuck a pin through a carrot. When I pulled the pin out, it had a hole.',
 'sentence2': 'The carrot had a hole.',
 'label': 1,
 'idx': 0}

**Вопрос:** что означают поля `sentence1`, `sentence2` и `label` в загруженном примере?

#### Токенизация

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

**Задание 2:** выведите содержимое переменной `tokenizer`. Какие атрибуты отвечают за словарь модели и за правила разбиения текста?

In [6]:
task_to_keys = {
    "cola": ("sentence", None),
    "mnli": ("premise", "hypothesis"),
    "mnli-mm": ("premise", "hypothesis"),
    "mrpc": ("sentence1", "sentence2"),
    "qnli": ("question", "sentence"),
    "qqp": ("question1", "question2"),
    "rte": ("sentence1", "sentence2"),
    "sst2": ("sentence", None),
    "stsb": ("sentence1", "sentence2"),
    "wnli": ("sentence1", "sentence2"),
}

sentence1_key, sentence2_key = task_to_keys[task]

def preprocess_function(examples):
    if sentence2_key is None:
        return tokenizer(examples[sentence1_key], truncation=True)
    return tokenizer(examples[sentence1_key], examples[sentence2_key], truncation=True)

encoded_dataset = dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/635 [00:00<?, ? examples/s]

Map:   0%|          | 0/71 [00:00<?, ? examples/s]

Map:   0%|          | 0/146 [00:00<?, ? examples/s]

**Задание 3:** объясните, зачем в функции `preprocess_function` нужна проверка `if sentence2_key is None`. Для каких задач из словаря `task_to_keys` это условие будет истинным? Что произойдет, если для задачи с двумя предложениями вызвать `tokenizer` только с одним аргументом?

#### Загрузка метрики

In [11]:
!pip install evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00


In [12]:
import evaluate

metric = evaluate.load('glue', task)

#### Тонкая настройка модели

In [13]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

num_labels = 3 if task.startswith("mnli") else 1 if task=="stsb" else 2
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels)

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


**Вопрос:** почему для задачи WNLI мы используем `num_labels=2`?

In [14]:
# выбор метрики зависит от задачи
metric_name = "pearson" if task == "stsb" else "matthews_correlation" if task == "cola" else "accuracy"
model_name = model_checkpoint.split("/")[-1]

args = TrainingArguments(
    f"{model_name}-finetuned-{task}",
    eval_strategy = "epoch",     # частота оценки на валидации
    save_strategy = "epoch",      # частота сохранения чекпоинтов
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model=metric_name,
    push_to_hub=True,
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # для классификации берем argmax, для регрессии (stsb) — первый элемент
    if task != "stsb":
        predictions = np.argmax(predictions, axis=1)
    else:
        predictions = predictions[:, 0]
    return metric.compute(predictions=predictions, references=labels)

**Задание 4:** параметры `eval_strategy` и `save_strategy` определяют, как часто оценивается и сохраняется модель. Зачем их устанавливать одинаковыми при использовании `load_best_model_at_end=True`?

In [17]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

validation_key = "validation_mismatched" if task == "mnli-mm" else "validation_matched" if task == "mnli" else "validation"
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset[validation_key],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [18]:
# запуск обучения
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.685953,0.563380
2,No log,0.694550,0.309859
3,No log,0.694884,0.366197
4,No log,0.695315,0.380282
5,No log,0.694922,0.535211


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=200, training_loss=0.6973487854003906, metrics={'train_runtime': 193.063, 'train_samples_per_second': 16.445, 'train_steps_per_second': 1.036, 'total_flos': 127699306274880.0, 'train_loss': 0.6973487854003906, 'epoch': 5.0})

#### Оценка модели

In [19]:
results = trainer.evaluate()
results

{'eval_loss': 0.6859622597694397,
 'eval_accuracy': 0.5633802816901409,
 'eval_runtime': 0.4739,
 'eval_samples_per_second': 149.833,
 'eval_steps_per_second': 10.552,
 'epoch': 5.0}

**Вопрос:** посмотрите на полученные метрики. Например, `eval_accuracy` около 0.56. Это хороший результат для задачи WNLI? Какая точность была бы у случайного классификатора?

---

# Домашнее задание

#### **Задание 1:** возьмите задачу `"rte"` (Recognizing Textual Entailment) из GLUE и дообучите на ней модель `bert-base-uncased`. Сравните полученные метрики с результатами на `wnli`

**Вопросы:**
- Какая из задач оказалась сложнее для модели?
- Сколько эпох потребовалось для достижения лучшего результата?
- Совпадают ли метрики с лидербордом GLUE?

#### **Задание 2:** выберите любую задачу классификации из GLUE (кроме `wnli`) и сравните две модели: `bert-base-uncased` и `roberta-base`. Для каждой модели:

1. Загрузите и токенизируйте данные
2. Обучите в течение 3 эпох
3. Зафиксируйте лучшую метрику на валидации

**Вопросы:**
- Какая модель показала себя лучше?
- Есть ли разница в скорости обучения?
- Как отличаются размеры моделей?

#### **Задание 3:** используйте `bert-base-multilingual-cased` для задачи `"xnli"` (не GLUE, а его мультиязычная версия). Выберите два языка (например, русский и немецкий) и сравните, насколько хорошо модель справляется с каждым из них до и после дообучения только на английском.